In [175]:
import os
import json
import random
import pickle as pkl
import networkx as nx
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.svm import SVC
from fastnode2vec import Node2Vec, Graph
from gensim.models import Word2Vec
from datasets import load_metric
from utils import get_texts_for_ptype

import xgboost as xgb
import numpy as np
import scipy as sp
from sklearn.linear_model import SGDClassifier
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer
from datasets import load_metric

In [2]:
study_design = 'is_rct'

In [3]:
neg_to_pos_ratio = 1

In [15]:
def get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='train'):
    
    pos_pmids = pmids[study_design]['pos'][split]
    pos_embeddings = {}

    for pmid in tqdm(pos_pmids):
        try:
            openAlexID = pmid_to_openAlex[str(pmid)]
            embedding = model.get_vector(openAlexID)
            pos_embeddings[pmid] = embedding
        except:
            continue

    neg_pmids = pmids[study_design]['neg'][split]
    neg_embeddings = {}
    for pmid in tqdm(neg_pmids):
        try:
            openAlexID = pmid_to_openAlex[str(pmid)]
            embedding = model.get_vector(openAlexID)
            neg_embeddings[pmid] = embedding
        except:
            continue
    return pos_embeddings, neg_embeddings

In [50]:
def get_texts_for_ptype_pmid(ptype, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True):
    inputs = "titles_abstracts"

    filename_pos_human = "../data/pubmed_articles_pos_human.csv"
    filename_neg_human = "../data/pubmed_articles_neg_human.csv"

    with open("../data/cleaned_pos_neg_train_val_test_dataset_ids.pkl", "rb") as f:
        prop_typ_id_dict = pkl.load(f)

    pos_data_human = pd.read_csv(filename_pos_human, chunksize=100000)
    neg_data_human = pd.read_csv(filename_neg_human, chunksize=100000)
    sr_pos_train_ids = prop_typ_id_dict[ptype]["pos"]["train"]
    sr_neg_train_ids = prop_typ_id_dict[ptype]["neg"]["train"]

    sr_pos_val_ids = prop_typ_id_dict[ptype]["pos"]["val"]
    sr_neg_val_ids = prop_typ_id_dict[ptype]["neg"]["val"]

    sr_pos_test_ids = prop_typ_id_dict[ptype]["pos"]["test"]
    sr_neg_test_ids = prop_typ_id_dict[ptype]["neg"]["test"]

    sr_pos_train_set = pd.DataFrame()
    sr_pos_val_set = pd.DataFrame()
    sr_pos_test_set = pd.DataFrame()

    for pos_chunk in tqdm(pos_data_human):

        indices = pos_chunk.index
        indices = list(indices)

        pos_train_indices = list(
            set(indices).intersection(set(sr_pos_train_ids))
        )
        pos_val_indices = list(set(indices).intersection(set(sr_pos_val_ids)))
        pos_test_indices = list(set(indices).intersection(set(sr_pos_test_ids)))

        sr_pos_train_subset = pos_chunk.loc[pos_train_indices]
        if sr_pos_train_set.empty:
            sr_pos_train_set = pd.DataFrame(columns=sr_pos_train_set.columns)
        sr_pos_train_set = pd.concat([sr_pos_train_set, sr_pos_train_subset])

        sr_pos_val_subset = pos_chunk.loc[pos_val_indices]
        if sr_pos_val_set.empty:
            sr_pos_val_set = pd.DataFrame(columns=sr_pos_val_set.columns)
        sr_pos_val_set = pd.concat([sr_pos_val_set, sr_pos_val_subset])

        sr_pos_test_subset = pos_chunk.loc[pos_test_indices]
        if sr_pos_test_set.empty:
            sr_pos_test_set = pd.DataFrame(columns=sr_pos_test_set.columns)
        sr_pos_test_set = pd.concat([sr_pos_test_set, sr_pos_test_subset])

        sr_pos_train_set[ptype] = int(1)
        sr_pos_val_set[ptype] = int(1)
        sr_pos_test_set[ptype] = int(1)

    del pos_data_human
    del sr_pos_train_subset
    del pos_chunk
    del indices
    del sr_pos_train_ids

    sr_neg_train_ids_sample = random.sample(
        list(sr_neg_train_ids), round(len(sr_pos_train_set) * neg_to_pos_ratio)
    )

    sr_neg_train_set = pd.DataFrame()
    sr_neg_val_set = pd.DataFrame()
    sr_neg_test_set = pd.DataFrame()

    for neg_chunk in tqdm(neg_data_human):

        indices = neg_chunk.index
        indices = list(indices)
        neg_train_indices = list(
           set(indices).intersection(set(sr_neg_train_ids_sample))
        )
        neg_val_indices = list(set(indices).intersection(set(sr_neg_val_ids)))
        neg_test_indices = list(set(indices).intersection(set(sr_neg_test_ids)))
        sr_neg_subset = neg_chunk.loc[indices]

        sr_neg_train_subset = neg_chunk.loc[neg_train_indices]
        if sr_neg_train_set.empty:
            sr_neg_train_set = pd.DataFrame(columns=sr_neg_train_set.columns)
        sr_neg_train_set = pd.concat([sr_neg_train_set, sr_neg_train_subset])

        sr_neg_val_subset = neg_chunk.loc[neg_val_indices]
        if sr_neg_val_set.empty:
            sr_neg_val_set = pd.DataFrame(columns=sr_neg_val_set.columns)
        sr_neg_val_set = pd.concat([sr_neg_val_set, sr_neg_val_subset])

        sr_neg_test_subset = neg_chunk.loc[neg_test_indices]
        if sr_neg_test_set.empty:
            sr_neg_test_set = pd.DataFrame(columns=sr_neg_test_set.columns)
        sr_neg_test_set = pd.concat([sr_neg_test_set, sr_neg_test_subset])

    sr_neg_train_set[ptype] = int(0)
    sr_neg_val_set[ptype] = int(0)
    sr_neg_test_set[ptype] = int(0)

    del neg_data_human
    del sr_neg_subset
    del neg_chunk
    del indices

    sr_train_data = pd.concat([sr_neg_train_set, sr_pos_train_set]).sample(
        frac=1
    )
    del sr_neg_train_set
    del sr_pos_train_set
    sr_val_data = pd.concat([sr_neg_val_set, sr_pos_val_set]).sample(frac=1)
    del sr_neg_val_set
    del sr_pos_val_set
    sr_test_data = pd.concat([sr_neg_test_set, sr_pos_test_set]).sample(frac=1)
    del sr_neg_test_set
    del sr_pos_test_set
    if inputs == "titles_abstracts":
        train_set = {
            pmid: {"X": f"{train_title} {train_abstract}", "y": y}
            for pmid, train_title, train_abstract, y in zip(
                sr_train_data['pmid'], sr_train_data["title"], sr_train_data["abstract_plaintext"], sr_train_data[ptype]
            )
            
        }
        val_set = {
            pmid: {"X": f"{val_title} {val_abstract}", "y": y}
            for pmid, val_title, val_abstract, y in zip(
                sr_val_data['pmid'], sr_val_data["title"], sr_val_data["abstract_plaintext"], sr_train_data[ptype]
            )
        }
        test_set = {
            pmid: {"X": f"{test_title} {test_abstract}", "y": y}
            for pmid, test_title, test_abstract, y in zip(
                sr_test_data['pmid'], sr_test_data["title"], sr_test_data["abstract_plaintext"], sr_train_data[ptype]
            )
        }
    del sr_train_data
    del sr_val_data
    del sr_test_data
    
    return train_set, val_set, test_set

In [187]:
class BatchVectorizerXGBoostConcat:
    """Custom vectorizer and classifier class to process data in
    batches.  Useful when there are RAM and CPU limitations.

    Args:
        BaseEstimator (Estimator): base class for estimator
    """

    def __init__(
        self,
        objective,
        max_depth,
        gamma,
        reg_alpha,
        reg_lambda,
        colsample_bytree,
        min_child_weight,
        n_estimators,
        seed,
        use_label_encoder=False,
        base_score=None,
        booster=None,
        colsample_bylevel=None,
        colsample_bynode=None,
        enable_categorical=None,
        gpu_id=None,
        importance_type=None,
        interaction_constraints=None,
        learning_rate=None,
        max_delta_step=None,
        *args,
        **kwargs
    ):
        """intialization of class

        Args:
            vectorizer (Vectorizer): vectorizer to turn text into vectors
            batch_size (int): size of each batch used for training
        """

        self.vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english", n_features=1000)
        self.batch_size = 100000
        self.xgbclass = xgb.XGBClassifier(
            objective=objective,
            max_depth=max_depth,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            n_estimators=n_estimators,
            seed=seed,
            base_score=base_score,
            booster=booster,
            colsample_bylevel=colsample_bylevel,
            colsample_bynode=colsample_bynode,
            enable_categorical=enable_categorical,
            importance_type=importance_type,
            interaction_constraints=interaction_constraints,
            learning_rate=learning_rate,
            max_delta_step=max_delta_step
        )
        # self._estimator_type = "classifier"

    def fit(self, X_texts, X_node_embeds, y):
        """fit the vectorizer and the classifier

        Args:
            X_texts (pd.Series): input texts
            X_node_embeds (pd.Series): input node embeddings
            y (pd.Series): labels
        """

        # check whether vectorize and modelr has been fitted
        vec_fitted = False
        model_fitted = False

        # number of batches to be processed
        if len(y) >= self.batch_size:
            num_batches = len(y) // self.batch_size
        else:
            num_batches = 1
            
        for X_text_chunk, X_node_embeds_chunk, y_chunk in zip(
            np.array_split(X_texts, num_batches), np.array_split(X_node_embeds, num_batches), 
            np.array_split(y, num_batches)
        ):

            # convert text chunk to unicode
            X_text_chunk = X_text_chunk.values.astype("U")

            if not vec_fitted:
                X_text_trans = self.vectorizer.fit_transform(X_text_chunk)
                vec_fitted = True
            else:
                X_text_trans = self.vectorizer.transform(X_text_chunk)
            del X_text_chunk
            
            X_text_trans = X_text_trans.todense()
            X_concat = np.hstack([np.vstack(X_text_trans), np.vstack(X_node_embeds_chunk)])
            # xg_chunk = xgb.DMatrix(X_trans, label=y_chunk)
            y_chunk = np.array(y_chunk).reshape(-1)
            if model_fitted:
                self.xgbclass.fit(
                    X_concat, y_chunk, xgb_model=self.xgbclass.get_booster()
                )
            else:
                self.xgbclass.fit(X_concat, y_chunk)
                model_fitted = True
            # if os.path.exists("model.model"):
            #     self.model = xgb.train(params, xg_chunk, 30, xgb_model="model.model")
            # else:
            #     self.model = xgb.train(params, xg_chunk, 30)
            # self.model.save_model("model.model")

        return self
    
    def get_params(self, deep):
        return self.xgbclass.get_params(deep=deep)

    def set_params(self, **parameters):
        return self.xgbclass.set_params(**parameters)

    def predict(self, X_texts, X_node_embeds):

        X_unicode = X_texts.values.astype("U")
        X_text_trans = self.vectorizer.transform(X_unicode).todense()
        X_concat = np.hstack([np.vstack(X_text_trans), np.vstack(X_node_embeds)])
        # xg_test = xgb.DMatrix(X_trans)
        predictions = np.array(list(map(int, np.round(self.xgbclass.predict(X_concat)))))
        return predictions

        # predictions = self.clf.predict(X_trans)
        # return predictions


In [5]:
neg_to_pos_ratio = 1
results_file = "results_node2vec_gbdt_sample.json"
metrics_file = "results_crossval_node2vec_gbdt_sample.json"
MODEL_DIR = "../models"
EMBEDDING_FILENAME = os.path.join(MODEL_DIR, 'node_model.pkl')
PMIDS = '../data/pmids.pkl'

In [6]:
with open(results_file, "r") as f:
    results_list = json.load(f)

In [7]:
with open("../data/graph_lst.pkl", "rb") as f:
    graph_lst = pkl.load(f)

In [8]:
graph = Graph(graph_lst, directed=False, weighted=False)
node2vec = Node2Vec(graph, dim=10, walk_length=100, context=10, p=2.0, q=0.5, workers=5)  # Use temp_folder for big graphs
model = node2vec.wv.load_word2vec_format(EMBEDDING_FILENAME)

Reading graph: 100%|████████████████████████████| 39051/39051 [00:00<00:00, 1842282.00it/s]


In [9]:
with open("../data/pmid_to_openAlex.json") as f:
    pmid_to_openAlex = json.load(f)

In [10]:
with open(PMIDS, 'rb') as f:
    pmids = pkl.load(f)

In [16]:
pos_train_embeddings, neg_train_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='train')

100%|██████████████████████████████████████████████| 4000/4000 [00:00<00:00, 476354.80it/s]


In [22]:
pos_val_embeddings, neg_val_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='val')

100%|██████████████████████████████████████████████| 4000/4000 [00:00<00:00, 495034.55it/s]


In [23]:
pos_test_embeddings, neg_test_embeddings = get_embeddings(pmids, pmid_to_openAlex, study_design, model, split='test')

100%|██████████████████████████████████████████████| 4000/4000 [00:00<00:00, 523111.00it/s]


In [87]:
pos_train_pmids = list(pos_train_embeddings.keys())
neg_train_pmids = list(neg_train_embeddings.keys())
pos_val_pmids = list(pos_val_embeddings.keys())
neg_val_pmids = list(neg_val_embeddings.keys())
pos_test_pmids = list(pos_test_embeddings.keys())
neg_test_pmids = list(neg_test_embeddings.keys())

In [88]:
pmids_to_use = {
    "pos_train": pos_train_pmids,
    "neg_train": neg_train_pmids,
    "pos_val": pos_val_pmids,
    "neg_val": neg_val_pmids,
    "pos_test": pos_test_pmids,
    "neg_test": neg_test_pmids
}

In [51]:
train_set, val_set, test_set = get_texts_for_ptype(study_design, inputs="titles_abstracts", neg_to_pos_ratio=1, reset_index=True)

19it [00:25,  1.36s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
51it [01:07,  1.35s/it]/home/k2141255/.local/lib/python3.8/site-packages/tqdm/std.py:1182: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
86it [01:53,  1.32s/it]
57it [01:25,  1.50s/it]


In [52]:
train_set

{8547730: {'X': 'Aerosolized beta 2-agonists do not induce bronchial hyperreactivity in healthy adults. OBJECTIVE\n\n\nTo determine whether regular administration of beta 2-agonists could induce bronchial hyperreactivity in nonasthmatic volunteers.\nDESIGN\n\n\nA prospective, randomized, double-blind, placebo-controlled, crossover clinical trial of 2 weeks therapy with a 2-week washout period between each treatment period. Treatments were albuterol or matching placebo as 2 inhalations 4 times daily.\nSUBJECTS\n\n\nTen healthy, nonsmoking women 27-37 years old.\nSETTING\n\n\nGeneral clinical research center of a tertiary care university hospital.\nMAIN OUTCOME MEASURES\n\n\nBaseline spirometry and methacholine bronchoprovocation studies were performed immediately prior to, 12 hours following, and 1 and 2 weeks following each treatment period.\nRESULTS\n\n\nNo change was detected in either baseline spirometry or methacholine responsiveness.\nCONCLUSIONS\n\n\nThis suggests that if beta 2-

In [53]:
pos_train_embeddings

{322634: array([-0.8788641 ,  1.7343812 , -0.06494753, -0.35726056,  0.56858605,
         1.3932437 ,  0.5504034 ,  1.3589609 ,  0.66341525, -0.41077235],
       dtype=float32),
 30311848: array([-1.3395834 ,  2.8461277 ,  0.05220188, -0.691697  ,  1.112216  ,
         1.6458846 ,  1.161846  ,  1.0214937 ,  0.6641165 , -0.21733189],
       dtype=float32),
 6180562: array([-0.8298937 ,  0.10586957,  1.1504334 , -0.05321076,  0.655724  ,
         0.3369605 , -0.9016402 ,  1.139648  ,  0.50809246, -1.0579618 ],
       dtype=float32),
 9783191: array([-0.28745335,  0.34439135,  0.15730077, -0.4013368 ,  0.86991465,
        -0.13853249,  0.10794524,  0.30913177,  0.41648808, -0.45795357],
       dtype=float32),
 19072879: array([-1.0654026 ,  0.5475083 ,  2.289932  , -0.97365326,  0.32035133,
         0.6179357 , -1.1358683 ,  1.1393152 ,  1.6614417 , -1.734681  ],
       dtype=float32),
 27097736: array([-1.1853529 ,  2.0654645 ,  0.2922002 , -0.79428107,  0.68649584,
         1.0567605 , 

In [158]:
train_set_concat = {}
for pmid in pos_train_pmids:
    train_set_concat[pmid] = {}
    train_set_concat[pmid]["X_graph"] = pos_train_embeddings[pmid]
    train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
    train_set_concat[pmid]["y"] = 1

for pmid in neg_train_pmids:
    train_set_concat[pmid] = {}
    train_set_concat[pmid]["X_graph"] = neg_train_embeddings[pmid]
    train_set_concat[pmid]["X_text"] = train_set[pmid]["X"]
    train_set_concat[pmid]["y"] = 0

val_set_concat = {}
for pmid in pos_val_pmids:
    val_set_concat[pmid] = {}
    val_set_concat[pmid]["X_graph"] = pos_val_embeddings[pmid]
    val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
    val_set_concat[pmid]["y"] = 1

for pmid in neg_val_pmids:
    val_set_concat[pmid] = {}
    val_set_concat[pmid]["X_graph"] = neg_val_embeddings[pmid]
    val_set_concat[pmid]["X_text"] = val_set[pmid]["X"]
    val_set_concat[pmid]["y"] = 0

test_set_concat = {}
for pmid in pos_test_pmids:
    test_set_concat[pmid] = {} 
    test_set_concat[pmid]["X_graph"] = pos_test_embeddings[pmid]
    test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
    test_set_concat[pmid]["y"] = 1

for pmid in neg_test_pmids:
    test_set_concat[pmid] = {}
    test_set_concat[pmid]["X_graph"] = neg_test_embeddings[pmid]
    test_set_concat[pmid]["X_text"] = test_set[pmid]["X"]
    test_set_concat[pmid]["y"] = 0

AttributeError: 'numpy.ndarray' object has no attribute 'dense'

In [90]:
def get_texts_citations_labels(data_set):
    texts = []
    node_embeds = []
    labels = []
    for pmid in data_set:
        texts.append(data_set[pmid]["X_text"])
        node_embeds.append(data_set[pmid]["X_graph"])
        labels.append(data_set[pmid]["y"])
        
    return pd.DataFrame(
        {
            "texts": texts,
            "node_embeds": node_embeds,
            "labels": labels
        }
    )

In [91]:
train_df = get_texts_citations_labels(train_set_concat)
train_df = train_df.sample(frac=1)

In [96]:
val_df = get_texts_citations_labels(val_set_concat)
val_df = val_df.sample(frac=1)

In [99]:
test_df = get_texts_citations_labels(test_set_concat)
test_df = test_df.sample(frac=1)

In [101]:
len(train_df)

7437

In [102]:
len(val_df)

7370

In [100]:
len(test_df)

7406

In [103]:
np.concatenate

<function numpy.concatenate>

In [188]:
bvc = BatchVectorizerXGBoost(
    objective="binary:logistic",
    max_depth=18,
    gamma=0.4812937402086781, 
    reg_alpha=79.57847865988239,
    reg_lambda=45699.356747877806,
    colsample_bytree=0.8773943298431721,
    min_child_weight=8.370014632905807,
    n_estimators=180,
    seed=0,
)

In [189]:
bvc.fit(train_df["texts"], train_df["node_embeds"], train_df["labels"])

/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)


[14:06:58] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1645117766796/work/src/learner.cc:1115: Starting in XGBoost 1.3.0, the default evaluation metric used with the objective 'binary:logistic' was changed from 'error' to 'logloss'. Explicitly set eval_metric if you'd like to restore the old behavior.


In [181]:
metric_acc = load_metric("accuracy")
metric_p = load_metric("precision")
metric_r = load_metric("recall")
metric_f = load_metric("f1")

/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/datasets/load.py:752: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.16.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/datasets/load.py:752: FutureWarning: The repository for precision contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.16.1/metrics/precision/precision.py
You can avoid this message in future by passing the argument `trust_rem

In [198]:
labels = test_df["labels"]
predictions = bvc.predict(test_df["texts"], test_df["node_embeds"])

In [199]:
acc = metric_acc.compute(predictions=predictions, references=labels)['accuracy']
p = metric_p.compute(predictions=predictions, references=labels)['precision']
r = metric_r.compute(predictions=predictions, references=labels)['recall']
f = metric_f.compute(predictions=predictions, references=labels)['f1']

In [200]:
acc

0.7774777207669458

In [201]:
p

0.7931228861330327

In [202]:
f

0.7735019241341397

In [203]:
r

0.7548283261802575